[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Evaluate ReID Models

Reproduce standard **same-domain** person re-identification benchmarks for OSNet x1.0
on Market-1501 and MSMT17 using the [`reid`](https://github.com/roboflow/re-ID) package
(`ReIDModel` + `ReIDEvaluator`).

Each dataset is scored with the checkpoint trained on that same dataset
([torchreid model zoo](https://kaiyangzhou.github.io/deep-person-reid/MODEL_ZOO)).

| Dataset | Size | T4 runtime | Expected (euclidean) |
|---|---|---|---|
| Market-1501 | ~1.3 GB | ~10 min | R1 ~94.2 / mAP ~82.6 |
| MSMT17 | ~4 GB | ~40 min | R1 ~74.9 / mAP ~43.8 |

> **Why not the default checkpoint?** `ReIDModel.from_pretrained()` ships OSNet trained on
> MSMT17 with `combineall=True` (train+test combined). Evaluating that on MSMT17 leaks the
> test set (near-100%) and on Market-1501 only measures cross-domain transfer (~61 R1).
> Below we load each dataset's own same-domain checkpoint instead.

> **Runtime:** T4 GPU (`Runtime → Change runtime type`).


## Setup

Install **reid** from PyPI. Colab already ships CUDA PyTorch, which pip will reuse when it
satisfies the package requirement.


In [ ]:
!pip install -q --upgrade pip
!pip install -q reid matplotlib scikit-learn

In [ ]:
import zipfile
from pathlib import Path

import gdown
import torch

from reid import ReIDEvaluator, ReIDModel

ROOT = Path("..")
CKPT_DIR = ROOT / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | {device}")


def download_checkpoint(gdrive_id: str, filename: str) -> str:
    """Download an OSNet checkpoint from the torchreid model zoo (Google Drive)."""
    path = CKPT_DIR / filename
    if not path.exists():
        gdown.download(id=gdrive_id, output=str(path), quiet=False)
    return str(path)


def print_comparison(name, cos, euc, zoo_r1, zoo_map):
    """Print a cosine-vs-euclidean comparison table against model-zoo targets."""
    print(f"\n{name}: distance metric comparison")
    print(f"{'metric':<10}{'cosine':>10}{'euclidean':>12}{'model zoo':>12}")
    print("-" * 44)
    print(f"{'Rank-1':<10}{cos.rank1:>9.1f}%{euc.rank1:>11.1f}%{zoo_r1:>11.1f}%")
    print(f"{'mAP':<10}{cos.mean_average_precision:>9.1f}%{euc.mean_average_precision:>11.1f}%{zoo_map:>11.1f}%")
    print(f"{'Rank-5':<10}{cos.rank5:>9.1f}%{euc.rank5:>11.1f}%{'-':>12}")
    print(f"{'Rank-10':<10}{cos.rank10:>9.1f}%{euc.rank10:>11.1f}%{'-':>12}")
    print(f"{'mINP':<10}{cos.minp:>9.1f}%{euc.minp:>11.1f}%{'-':>12}")


## Market-1501

**Expected (OSNet x1.0, Market-trained, model zoo):** R1 ≈ 94.2 / mAP ≈ 82.6

The model-zoo protocol uses raw **euclidean** distance. We also report **cosine**
(L2-normalised embeddings, our default) so you can see the gap.

Market-1501 is a public research benchmark. This notebook downloads the public Google Drive
archive linked by the [dataset authors](https://zheng-lab-anu.github.io/Project/project_reid.html)
(~1.3 GB extracted). Review the dataset's terms before use. If Google Drive fails, use the
fallback cell below.


In [ ]:
MARKET_ZIP = ROOT / "Market-1501.zip"
MARKET_DIR = ROOT / "Market-1501-v15.09.15"

if not MARKET_DIR.exists():
    # Google Drive file ID for Market-1501-v15.09.15.zip
    gdown.download(id="0B8-rUzbwVRk0c054eEozWG9COHM", output=str(MARKET_ZIP), quiet=False)
    with zipfile.ZipFile(MARKET_ZIP, "r") as zf:
        zf.extractall(ROOT)
    print("Extracted to", MARKET_DIR)
else:
    print("Already downloaded.")


In [ ]:
# Fallback if gdown fails: paste your mirror URL:
# !wget -q -O ../Market-1501.zip "<YOUR_MIRROR_URL>"
# !unzip -q ../Market-1501.zip -d ..


In [ ]:
from reid import load_market1501

query_m, gallery_m = load_market1501(str(MARKET_DIR))
print(f"Market-1501: {len(query_m):,} query, {len(gallery_m):,} gallery")


In [ ]:
# OSNet x1.0 checkpoint trained on Market-1501 (model zoo).
market_ckpt = download_checkpoint("1vduhq5DpN2q1g4fYEZfPI17MJeh9qyrA", "osnet_x1_0_market1501.pth")
model_market = ReIDModel.from_pretrained(market_ckpt, architecture="osnet_x1_0")
evaluator_market = ReIDEvaluator(model_market, batch_size=256)

# Cosine (default): extract embeddings once.
result_market = evaluator_market.evaluate(query_m, gallery_m, distance="cosine", return_distmat=False)
# Euclidean (model-zoo protocol): re-score cached embeddings.
result_market_euc = evaluator_market.evaluate(
    query_m,
    gallery_m,
    distance="euclidean",
    return_distmat=False,
    query_embeddings=result_market.query_embeddings,
    gallery_embeddings=result_market.gallery_embeddings,
)

print_comparison("Market-1501", result_market.metrics, result_market_euc.metrics, 94.2, 82.6)

## MSMT17

**Expected (OSNet x1.0, MSMT-trained, model zoo):** R1 ≈ 74.9 / mAP ≈ 43.8 (euclidean; cosine also printed)

MSMT17 is distributed by Peking University NELVT for academic use only. Request the official
**MSMT17_V1** release after signing the
[MSMT17 release agreement](https://www.pkuvmc.com/agreement/RELEASE_AGREEMENT-MSMT17.pdf).
Do not use unofficial re-uploads until the archive can be verified against the official distribution.

After download, point `MSMT17_DIR` at the extracted root (must contain `test/`, `list_query.txt`,
and `list_gallery.txt`).


In [ ]:
# Official MSMT17_V1 only: request from NELVT after signing the release agreement.
# https://www.pkuvmc.com/agreement/RELEASE_AGREEMENT-MSMT17.pdf
MSMT17_DIR = ROOT / "MSMT17_V1"  # extracted official root

assert MSMT17_DIR.is_dir(), (
    f"MSMT17 not found at {MSMT17_DIR}. "
    "Request the official MSMT17_V1 release, extract it, then set MSMT17_DIR."
)
assert (MSMT17_DIR / "list_query.txt").is_file(), "Missing list_query.txt"
assert (MSMT17_DIR / "list_gallery.txt").is_file(), "Missing list_gallery.txt"
assert (MSMT17_DIR / "test").is_dir(), "Missing test/"
print("Using official MSMT17 at", MSMT17_DIR)


In [ ]:
from reid import load_msmt17

query_ms, gallery_ms = load_msmt17(str(MSMT17_DIR))
print(f"MSMT17: {len(query_ms):,} query, {len(gallery_ms):,} gallery")


In [ ]:
# Same-domain MSMT17 checkpoint. Skip the default combineall alias (test leakage).
msmt17_ckpt = download_checkpoint("112EMUfBPYeYg70w-syK6V6Mx8-Qb9Q1M", "osnet_x1_0_msmt17.pth")
model_msmt17 = ReIDModel.from_pretrained(msmt17_ckpt, architecture="osnet_x1_0")
evaluator_msmt17 = ReIDEvaluator(model_msmt17, batch_size=256)

# Cosine (default): extract embeddings once.
result_msmt17 = evaluator_msmt17.evaluate(query_ms, gallery_ms, distance="cosine", return_distmat=False)
# Euclidean (model-zoo protocol): re-score cached embeddings.
result_msmt17_euc = evaluator_msmt17.evaluate(
    query_ms,
    gallery_ms,
    distance="euclidean",
    return_distmat=False,
    query_embeddings=result_msmt17.query_embeddings,
    gallery_embeddings=result_msmt17.gallery_embeddings,
)

print_comparison("MSMT17", result_msmt17.metrics, result_msmt17_euc.metrics, 74.9, 43.8)

## Summary


In [ ]:
print(f"{'Dataset':<14}{'encoder':<22}{'distance':<12}{'mAP':>8}{'Rank-1':>9}{'Rank-5':>9}{'Rank-10':>9}{'mINP':>8}")
print("-" * 91)


def _row(name, encoder, dist, m):
    return (
        f"{name:<14}{encoder:<22}{dist:<12}{m.mean_average_precision:>7.1f}%{m.rank1:>8.1f}%"
        f"{m.rank5:>8.1f}%{m.rank10:>8.1f}%{m.minp:>7.1f}%"
    )


print(_row("Market-1501", "OSNet", "cosine", result_market.metrics))
print(_row("", "", "euclidean", result_market_euc.metrics))
print(_row("MSMT17", "OSNet", "cosine", result_msmt17.metrics))
print(_row("", "", "euclidean", result_msmt17_euc.metrics))
print("-" * 91)
print("Model-zoo targets (OSNet, euclidean): Market-1501 R1≈94.2 mAP≈82.6  |  MSMT17 R1≈74.9 mAP≈43.8")